# 00 — Data Fetching

Loads the pre-computed segment feature CSV and enriches it with two additional OSM-derived columns:
`public_transport_proximity_m` and `dominant_land_use_score`.

**Input:** `csv/features_all_boroughs_with_location_id.csv`  
**Output:** `csv/features_all_boroughs_with_location_id_augmented.csv`

**Existing CSV-backed features are preserved as-is.** Only the two new columns are computed here.


In [1]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

ox.settings.log_console = False
ox.settings.use_cache = True

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
REPO_ROOT = Path('..').resolve()
CSV_DIR = REPO_ROOT / 'csv'
BASE_FEATURES_CSV = CSV_DIR / 'features_all_boroughs_with_location_id.csv'
AUGMENTED_FEATURES_CSV = CSV_DIR / 'features_all_boroughs_with_location_id_augmented.csv'

# ── Borough definitions ───────────────────────────────────────────────────────
BOROUGHS = {
    'westminster':    'City of Westminster, London, UK',
    'islington':      'London Borough of Islington, London, UK',
    'hackney':        'London Borough of Hackney, London, UK',
    'tower_hamlets':  'London Borough of Tower Hamlets, London, UK',
    'southwark':      'London Borough of Southwark, London, UK',
}

# Explicit mapping from CSV borough values to query strings
BOROUGH_QUERIES = {v: v for v in BOROUGHS.values()}

# Only keep these boroughs from the input CSV
ALLOWED_BOROUGH_NAMES = [
    'City of Westminster, London, UK',
    'London Borough of Islington, London, UK',
    'London Borough of Hackney, London, UK',
    'London Borough of Tower Hamlets, London, UK',
    'London Borough of Southwark, London, UK',
]

# ── Helper: load and project base feature CSV ────────────────────────────────
def load_base_features(csv_path):
    """Load the pre-computed segment features and attach projected geometry.

    Only rows whose `borough` is in `ALLOWED_BOROUGH_NAMES` are kept.
    """
    base_df = pd.read_csv(csv_path)
    if 'geometry' not in base_df.columns:
        raise ValueError(f'Missing geometry column in {csv_path}')

    # Filter to allowed boroughs
    if 'borough' in base_df.columns:
        base_df = base_df[base_df['borough'].isin(ALLOWED_BOROUGH_NAMES)].reset_index(drop=True)
    else:
        print('  [WARNING] No `borough` column found in base CSV; keeping all rows.')

    geometry = gpd.GeoSeries.from_wkt(base_df['geometry'], crs=CRS_PROJECTED)
    base_df = base_df.drop(columns='geometry')
    return gpd.GeoDataFrame(base_df, geometry=geometry, crs=CRS_PROJECTED)

Config ready.


In [7]:
# ── Helper: load and project Mapillary lights ─────────────────────────────────
def load_mapillary_lights():
    """Load Mapillary CSV and return a projected GeoDataFrame of light points."""
    if not MAPILLARY_CSV.exists():
        print(f'  [WARNING] Mapillary CSV not found at {MAPILLARY_CSV}.')
        print('  lamp_count will be set to NaN for all segments.')
        return None
    df = pd.read_csv(MAPILLARY_CSV)
    # Normalise column names to lowercase
    df.columns = df.columns.str.lower().str.strip()
    # Accept 'lat'/'lon' or 'latitude'/'longitude'
    lat_col = next((c for c in df.columns if c.startswith('lat')), None)
    lon_col = next((c for c in df.columns if c.startswith('lon')), None)
    if lat_col is None or lon_col is None:
        raise ValueError(f'Could not find lat/lon columns in {MAPILLARY_CSV}. Found: {list(df.columns)}')
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs='EPSG:4326'
    ).to_crs(CRS_PROJECTED)
    print(f'  Loaded {len(gdf):,} Mapillary light points.')
    return gdf

In [6]:
# ── Helper: count points within buffer of each segment ────────────────────────
def count_points_in_buffer(edges_proj, points_gdf, buffer_m, col_name):
    """Add a column counting how many points fall within buffer_m of each edge."""
    if points_gdf is None:
        edges_proj[col_name] = np.nan
        return edges_proj
    buffers = edges_proj.geometry.buffer(buffer_m)
    counts = []
    for buf in buffers:
        counts.append(points_gdf[points_gdf.geometry.within(buf)].shape[0])
    edges_proj[col_name] = counts
    return edges_proj

In [4]:
# ── Helper: transit proximity (rail vs bus) ───────────────────────────────────
RAIL_TAGS = {
    'railway': ['subway_entrance', 'station', 'tram_stop'],
    'public_transport': 'stop_position',
}
BUS_TAGS = {
    'highway': 'bus_stop',
    'public_transport': 'platform',
}


def _nearest_dist(edges_proj, stops_gdf, col_name):
    """Add column: distance in metres to nearest stop (EPSG:27700 = metres)."""
    segment_points = edges_proj.geometry.apply(
        lambda geom: geom.interpolate(0.5, normalized=True)
        if geom.geom_type in ('LineString', 'MultiLineString')
        else geom.centroid
    )
    if stops_gdf is None or stops_gdf.empty:
        edges_proj[col_name] = np.nan
        return edges_proj

    stop_coords = np.array([(g.x, g.y) for g in stops_gdf.geometry])
    dists = []
    for mp in segment_points:
        d = np.sqrt(((stop_coords - [mp.x, mp.y]) ** 2).sum(axis=1)).min()
        dists.append(round(float(d), 2))
    edges_proj[col_name] = dists
    return edges_proj


def add_transit_dist(edges_proj, borough_query):
    """
    dist_to_rail_m : nearest tube/overground/DLR/Elizabeth line entrance
    dist_to_bus_m  : nearest bus stop or platform
    """
    def fetch_stops(tags, label):
        try:
            gdf = ox.features_from_place(borough_query, tags=tags)
            gdf['geometry'] = gdf.geometry.apply(
                lambda g: g.centroid if g.geom_type != 'Point' else g
            )
            gdf = gdf[['geometry']].to_crs(CRS_PROJECTED).drop_duplicates('geometry')
            print(f'  {label}: {len(gdf):,} stops found.')
            return gdf
        except Exception as e:
            print(f'  [WARNING] {label} fetch failed: {e}')
            return None

    print('  Fetching rail stops...')
    rail = fetch_stops(RAIL_TAGS, 'rail')
    print('  Fetching bus stops...')
    bus = fetch_stops(BUS_TAGS, 'bus')

    if rail is not None:
        edges_proj = _nearest_dist(edges_proj, rail, 'dist_to_rail_m')
    else:
        edges_proj['dist_to_rail_m'] = np.nan

    if bus is not None:
        edges_proj = _nearest_dist(edges_proj, bus, 'dist_to_bus_m')
    else:
        edges_proj['dist_to_bus_m'] = np.nan

    return edges_proj

In [ ]:
# ── Helper: dominant land use score in buffer ────────────────────────────────
COMMERCIAL_LANDUSE_TAGS = {
    'landuse': ['commercial', 'retail', 'mixed_use'],
    'building': ['commercial', 'retail', 'office', 'supermarket'],
}
RESIDENTIAL_LANDUSE_TAGS = {
    'landuse': ['residential'],
    'building': ['residential', 'apartments', 'house', 'terrace', 'detached'],
}


def _area_in_buffer(polygons_gdf, buffer_geom):
    """Return total m² of polygons intersecting a buffer geometry."""
    candidates = polygons_gdf[polygons_gdf.geometry.intersects(buffer_geom)]
    if candidates.empty:
        return 0.0
    return candidates.geometry.intersection(buffer_geom).area.sum()


def add_dominant_land_use_score(edges_proj, borough_query):
    """
    dominant_land_use_score:
     0.0 = fully residential-dominant
     0.5 = balanced or no land-use signal
     1.0 = fully commercial-dominant
    (Range mapped to 0..1 to avoid confusion with scaling later.)
    """
    def fetch_polygons(tags, label):
        attempts = 2
        for attempt in range(1, attempts + 1):
            try:
                gdf = ox.features_from_place(borough_query, tags=tags)
                gdf = gdf[gdf.geometry.type.isin(['Polygon', 'MultiPolygon'])]
                gdf = gdf[['geometry']].to_crs(CRS_PROJECTED)
                print(f'  {label}: {len(gdf):,} polygons found.')
                return gdf
            except Exception as e:
                print(f'  [WARNING] {label} fetch attempt {attempt} failed: {e}')
                if attempt == attempts:
                    return None
                else:
                    print('    Retrying...')

    print('  Fetching commercial landuse...')
    commercial = fetch_polygons(COMMERCIAL_LANDUSE_TAGS, 'commercial landuse')
    print('  Fetching residential landuse...')
    residential = fetch_polygons(RESIDENTIAL_LANDUSE_TAGS, 'residential landuse')

    scores = []
    for geom in edges_proj.geometry:
        buf = geom.buffer(BUFFER_SMALL)
        if commercial is None or residential is None:
            scores.append(np.nan)
            continue
        commercial_area = _area_in_buffer(commercial, buf)
        residential_area = _area_in_buffer(residential, buf)
        total_area = commercial_area + residential_area
        if total_area == 0:
            # No land-use signal -> balanced (mapped to 0.5 in 0..1 scale)
            scores.append(0.5)
        else:
            # Raw score in [-1, 1] where +1 = commercial, -1 = residential
            raw = (commercial_area - residential_area) / total_area
            # Map to [0, 1]: -1 -> 0, 0 -> 0.5, 1 -> 1
            mapped = (raw + 1.0) / 2.0
            scores.append(round(float(mapped), 4))

    edges_proj['dominant_land_use_score'] = scores
    return edges_proj

In [9]:
# ── Helper: residential flag ──────────────────────────────────────────────────
def add_residential_flag(edges_proj, borough_query):
    """1 if a residential landuse polygon overlaps the 50 m buffer, else 0."""
    print('  Fetching residential landuse...')
    try:
        residential = ox.features_from_place(
            borough_query,
            tags={'landuse': ['residential', 'apartments', 'housing']}
        )
        residential = residential[
            residential.geometry.type.isin(['Polygon', 'MultiPolygon'])
        ][['geometry']].to_crs(CRS_PROJECTED)
    except Exception as e:
        print(f'  [WARNING] Could not fetch residential landuse: {e}')
        edges_proj['residential_flag'] = np.nan
        return edges_proj

    flags = []
    for geom in edges_proj.geometry:
        buf = geom.buffer(BUFFER_SMALL)
        flag = int(residential[residential.geometry.intersects(buf)].shape[0] > 0)
        flags.append(flag)
    edges_proj['residential_flag'] = flags
    return edges_proj

In [10]:
# ── Helper: transit proximity ─────────────────────────────────────────────────
# London transit: bus stops, Tube/Overground/Elizabeth line stations, tram stops
TRANSIT_TAGS = {
    'highway': 'bus_stop',
    'railway': ['station', 'subway_entrance', 'tram_stop', 'halt'],
    'public_transport': ['stop_position', 'platform'],
}

def add_transit_dist(edges_proj, borough_query):
    """Distance in metres to the nearest transit stop (EPSG:27700 is in metres)."""
    print('  Fetching transit stops...')
    try:
        stops = ox.features_from_place(borough_query, tags=TRANSIT_TAGS)
        stops['geometry'] = stops.geometry.apply(
            lambda g: g.centroid if g.geom_type != 'Point' else g
        )
        stops = stops[['geometry']].to_crs(CRS_PROJECTED)
        stops = stops.drop_duplicates(subset='geometry')
    except Exception as e:
        print(f'  [WARNING] Could not fetch transit stops: {e}')
        edges_proj['transit_dist_m'] = np.nan
        return edges_proj

    # For each segment midpoint, find nearest stop
    midpoints = edges_proj.geometry.interpolate(0.5, normalized=True)
    stop_geoms = np.array([(g.x, g.y) for g in stops.geometry])
    dists = []
    for mp in midpoints:
        if len(stop_geoms) == 0:
            dists.append(np.nan)
        else:
            d = np.sqrt(((stop_geoms - [mp.x, mp.y]) ** 2).sum(axis=1)).min()
            dists.append(round(float(d), 2))  # already metres in EPSG:27700
    edges_proj['transit_dist_m'] = dists
    return edges_proj

In [9]:
# ── Main enrichment loop ─────────────────────────────────────────────────────
base_features = load_base_features(BASE_FEATURES_CSV)
base_features['_row_order'] = np.arange(len(base_features))
base_features['public_transport_proximity_m'] = np.nan
base_features['dominant_land_use_score'] = np.nan

for borough_name, borough_features in base_features.groupby('borough', sort=False):
    borough_query = BOROUGH_QUERIES.get(borough_name)
    if borough_query is None:
        print(f'\n[WARNING] No OSM query configured for {borough_name!r}; leaving new columns as NaN.')
        continue

    print(f'\n══ {borough_name.upper()} ══')
    working = borough_features.copy()
    working = add_transit_dist(working, borough_query)
    working['public_transport_proximity_m'] = working[['dist_to_rail_m', 'dist_to_bus_m']].min(axis=1)
    working = working.drop(columns=['dist_to_rail_m', 'dist_to_bus_m'])

    working = add_dominant_land_use_score(working, borough_query)

    base_features.loc[working.index, 'public_transport_proximity_m'] = working['public_transport_proximity_m']
    base_features.loc[working.index, 'dominant_land_use_score'] = working['dominant_land_use_score']

base_features = base_features.sort_values('_row_order').drop(columns='_row_order')

if 'geometry' in base_features.columns:
    base_features['geometry'] = base_features.geometry.to_wkt()

AUGMENTED_FEATURES_CSV.parent.mkdir(parents=True, exist_ok=True)
base_features.to_csv(AUGMENTED_FEATURES_CSV, index=False)
print(f'\nSaved → {AUGMENTED_FEATURES_CSV.name}  ({len(base_features):,} rows)')


══ LONDON BOROUGH OF ISLINGTON, LONDON, UK ══
  Fetching rail stops...
  rail: 406 stops found.
  Fetching bus stops...
  bus: 442 stops found.
  Fetching commercial landuse...
  commercial landuse: 729 polygons found.
  Fetching residential landuse...
  residential landuse: 9,882 polygons found.

══ LONDON BOROUGH OF TOWER HAMLETS, LONDON, UK ══
  Fetching rail stops...
  rail: 607 stops found.
  Fetching bus stops...
  bus: 565 stops found.
  Fetching commercial landuse...
  commercial landuse: 1,125 polygons found.
  Fetching residential landuse...
  residential landuse: 13,289 polygons found.

══ LONDON BOROUGH OF HACKNEY, LONDON, UK ══
  Fetching rail stops...
  rail: 370 stops found.
  Fetching bus stops...
  bus: 511 stops found.
  Fetching commercial landuse...
  commercial landuse: 1,016 polygons found.
  Fetching residential landuse...
  residential landuse: 9,068 polygons found.

══ LONDON BOROUGH OF SOUTHWARK, LONDON, UK ══
  Fetching rail stops...
  rail: 457 stops found.

In [8]:
# ── Validation summary ────────────────────────────────────────────────────────
print('\n── Augmented data summary ──')
if AUGMENTED_FEATURES_CSV.exists():
    df = pd.read_csv(AUGMENTED_FEATURES_CSV)
    print(f'Loaded {len(df):,} rows from {AUGMENTED_FEATURES_CSV.name}')
    print(f"Columns: {', '.join(df.columns)}")
    nan_pct = df[['public_transport_proximity_m', 'dominant_land_use_score']].isna().mean().mul(100).round(1)
    print('\nMissing values in new columns:')
    print(nan_pct.to_string())
else:
    print(f'{AUGMENTED_FEATURES_CSV} not found')


── Augmented data summary ──
Loaded 61,784 rows from features_all_boroughs_with_location_id_augmented.csv
Columns: lighting, visibility, connectivity, enclosure, borough, latitude, longitude, location_id, geometry, public_transport_proximity_m, dominant_land_use_score

Missing values in new columns:
public_transport_proximity_m    41.8
dominant_land_use_score         57.6
